# Limpieza y transformación de datos — Cafe Sales (Dirty Data for Cleaning Training)

**Actividad individual — Análisis y visualización de la información**
**Autor:** Jorge Isaac Quintero Carreón (219515362)

Objetivo: aplicar técnicas de limpieza y transformación de datos con Python sobre una base de datos diseñada para contener problemas de calidad (`dirty_cafe_sales.csv`), documentar los problemas encontrados y exportar una versión limpia lista para análisis.


## 1. Carga e inspección de la base

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dirty_cafe_sales.csv')
print('Dimensiones:', df.shape)
df.head(10)


Dimensiones: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [2]:
df.dtypes


Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

In [3]:
# Todas las columnas se cargan como texto (object), incluso las que deberían ser
# numéricas (Quantity, Price Per Unit, Total Spent) o de fecha (Transaction Date).
# Esto ya es un primer indicio de un problema de "tipos de datos incorrectos".
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
# Conteo de valores nulos "reales" (NaN) por columna
df.isna().sum()


Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [5]:
# El dataset también usa los textos literales 'ERROR' y 'UNKNOWN' como marcadores
# de datos faltantes/corruptos en lugar de dejar la celda vacía. Los contamos aparte.
for col in df.columns:
    n_error = (df[col] == 'ERROR').sum()
    n_unknown = (df[col] == 'UNKNOWN').sum()
    print(f'{col:20s} ERROR={n_error:5d}  UNKNOWN={n_unknown:5d}')


Transaction ID       ERROR=    0  UNKNOWN=    0
Item                 ERROR=  292  UNKNOWN=  344
Quantity             ERROR=  170  UNKNOWN=  171
Price Per Unit       ERROR=  190  UNKNOWN=  164
Total Spent          ERROR=  164  UNKNOWN=  165
Payment Method       ERROR=  306  UNKNOWN=  293
Location             ERROR=  358  UNKNOWN=  338
Transaction Date     ERROR=  142  UNKNOWN=  159


In [6]:
# Duplicados
print('Filas duplicadas (todas las columnas):', df.duplicated().sum())
print('Transaction ID duplicados:', df['Transaction ID'].duplicated().sum())


Filas duplicadas (todas las columnas): 0
Transaction ID duplicados: 0


In [7]:
# Valores únicos en columnas categóricas, para revisar errores de formato
# (mayúsculas/minúsculas distintas, espacios, escrituras alternativas, etc.)
print('Item:', df['Item'].unique())
print('Payment Method:', df['Payment Method'].unique())
print('Location:', df['Location'].unique())


Item: <StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str
Payment Method: <StringArray>
['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR', nan]
Length: 6, dtype: str
Location: <StringArray>
['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR']
Length: 5, dtype: str


## 2. Problemas de calidad identificados

1. **Valores faltantes**: mezclados en tres formas distintas — celdas vacías (`NaN`), y los textos `'ERROR'` y `'UNKNOWN'` usados como marcadores de dato corrupto/desconocido. Afectan a todas las columnas excepto `Transaction ID`.
2. **Duplicados**: no se encontraron filas duplicadas ni `Transaction ID` repetidos.
3. **Errores de formato**: las categorías de `Item`, `Payment Method` y `Location` están escritas de forma consistente (sin variaciones de mayúsculas/minúsculas ni sinónimos); el problema de formato principal es que `'ERROR'`/`'UNKNOWN'` aparecen mezclados como si fueran categorías válidas.
4. **Valores atípicos**: `Quantity` (1–5) y `Price Per Unit` (1.0–5.0) se mueven en rangos acotados y razonables para una cafetería; no se detectaron valores atípicos extremos una vez convertidos a numérico.
5. **Tipos de datos incorrectos**: `Quantity`, `Price Per Unit` y `Total Spent` deberían ser numéricos y `Transaction Date` debería ser fecha, pero las cuatro columnas se cargan como texto por la mezcla de `'ERROR'`/`'UNKNOWN'` con valores numéricos/fechas.

Un hallazgo clave para la limpieza: cada `Item` tiene un `Price Per Unit` fijo (por ejemplo, Coffee siempre cuesta 2.0, Cookie siempre 1.0), y `Total Spent = Quantity × Price Per Unit` se cumple en el 100% de las filas donde los tres campos son válidos. Esta relación se aprovecha para reconstruir valores faltantes en lugar de eliminarlos o imputarlos con la media.


## 3. Limpieza y transformación

### 3.1 Estandarizar los marcadores de valores faltantes

In [8]:
# 'ERROR' y 'UNKNOWN' se tratan como equivalentes a un valor faltante real,
# para poder usar las herramientas estándar de pandas (isna, fillna, etc.)
df_clean = df.replace({'ERROR': np.nan, 'UNKNOWN': np.nan})


### 3.2 Corregir los tipos de datos

In [9]:
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean['Price Per Unit'] = pd.to_numeric(df_clean['Price Per Unit'], errors='coerce')
df_clean['Total Spent'] = pd.to_numeric(df_clean['Total Spent'], errors='coerce')
df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'], errors='coerce')
df_clean.dtypes


Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

### 3.3 Reconstruir valores faltantes usando las relaciones entre columnas

En vez de imputar directamente con la media o eliminar filas, se aprovechan dos relaciones deterministas presentes en los datos:

- `Price Per Unit` depende únicamente del `Item` (cada producto tiene un precio fijo).
- `Total Spent = Quantity × Price Per Unit`.

Con esto se recupera información real en lugar de inventar valores promedio, lo que reduce el riesgo de distorsionar el análisis (ver tabla de riesgos en el README).


In [10]:
# Mapa Item -> Precio, construido solo con filas donde ambos valores son válidos
item_price = (df_clean.dropna(subset=['Item', 'Price Per Unit'])
                        .groupby('Item')['Price Per Unit']
                        .agg(lambda x: x.mode()[0]))
item_price


Item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5
Name: Price Per Unit, dtype: float64

In [11]:
# Precios que identifican de forma única a un solo producto
# (3.0 y 4.0 son ambiguos: Cake/Juice y Sandwich/Smoothie comparten precio,
# por lo que NO se usan para inferir el Item, solo los precios únicos)
price_item = {v: k for k, v in item_price.items()
              if (item_price == v).sum() == 1}
price_item


{2.0: 'Coffee', 1.0: 'Cookie', 5.0: 'Salad', 1.5: 'Tea'}

In [12]:
n_price_from_item = 0
n_item_from_price = 0

for idx, row in df_clean.iterrows():
    item, price = row['Item'], row['Price Per Unit']

    if pd.isna(price) and pd.notna(item) and item in item_price.index:
        df_clean.at[idx, 'Price Per Unit'] = item_price[item]
        n_price_from_item += 1

    if pd.isna(item) and pd.notna(price) and price in price_item:
        df_clean.at[idx, 'Item'] = price_item[price]
        n_item_from_price += 1

print('Price Per Unit reconstruido a partir de Item:', n_price_from_item)
print('Item reconstruido a partir de Price Per Unit:', n_item_from_price)


Price Per Unit reconstruido a partir de Item: 479
Item reconstruido a partir de Price Per Unit: 468


In [13]:
n_qty = n_price = n_total = 0

for idx, row in df_clean.iterrows():
    qty, price, total = row['Quantity'], row['Price Per Unit'], row['Total Spent']

    if pd.isna(qty) and pd.notna(price) and pd.notna(total) and price != 0:
        df_clean.at[idx, 'Quantity'] = round(total / price)
        n_qty += 1
    elif pd.isna(price) and pd.notna(qty) and pd.notna(total) and qty != 0:
        df_clean.at[idx, 'Price Per Unit'] = round(total / qty, 2)
        n_price += 1
    elif pd.isna(total) and pd.notna(qty) and pd.notna(price):
        df_clean.at[idx, 'Total Spent'] = qty * price
        n_total += 1

print('Quantity reconstruida (Total / Price):', n_qty)
print('Price Per Unit reconstruido (Total / Quantity):', n_price)
print('Total Spent reconstruido (Quantity * Price):', n_total)


Quantity reconstruida (Total / Price): 456
Price Per Unit reconstruido (Total / Quantity): 48
Total Spent reconstruido (Quantity * Price): 479


### 3.4 Eliminar filas irrecuperables

Tras la reconstrucción, algunas filas siguen sin `Quantity`, `Price Per Unit` o `Total Spent` porque no había suficiente información cruzada para calcularlos (por ejemplo, faltan los tres campos numéricos a la vez, o falta `Item` y dos de los tres numéricos). Como el monto de la venta no puede determinarse ni aproximarse de forma confiable, esas filas se eliminan en vez de rellenarse con un supuesto arbitrario.


In [14]:
core_missing = df_clean[['Quantity', 'Price Per Unit', 'Total Spent']].isna().any(axis=1)
print('Filas eliminadas por datos numéricos irrecuperables:', core_missing.sum())

df_clean = df_clean.loc[~core_missing].copy()
df_clean['Quantity'] = df_clean['Quantity'].astype(int)
print('Filas restantes:', len(df_clean))


Filas eliminadas por datos numéricos irrecuperables: 26
Filas restantes: 9974


### 3.5 Tratar el resto de los faltantes

- `Item`: los casos que aún no se pudieron inferir (precio ambiguo entre dos productos, o item y precio ambos ausentes) se marcan explícitamente como `'Desconocido'` en vez de asignarles un producto al azar.
- `Payment Method` y `Location`: tienen una proporción muy alta de valores faltantes (~32 % y ~40 %). Imputar con la moda introduciría un sesgo importante hacia la categoría más frecuente, así que se mantienen como filas válidas y se marcan como `'Desconocido'`, dejando claro en los datos que ese dato no se conoce (en vez de ocultarlo con un valor inventado).
- `Transaction Date`: no existe ninguna otra columna de la que se pueda derivar la fecha, así que las fechas faltantes se dejan como valores nulos (`NaT`) en vez de imputarse; quedan disponibles para análisis que no dependan de la fecha y se excluyen automáticamente de los análisis temporales.


In [15]:
n_item_unknown = df_clean['Item'].isna().sum()
n_payment_unknown = df_clean['Payment Method'].isna().sum()
n_location_unknown = df_clean['Location'].isna().sum()
n_date_missing = df_clean['Transaction Date'].isna().sum()

df_clean['Item'] = df_clean['Item'].fillna('Desconocido')
df_clean['Payment Method'] = df_clean['Payment Method'].fillna('Desconocido')
df_clean['Location'] = df_clean['Location'].fillna('Desconocido')
# Transaction Date se deja como NaT intencionalmente (no se imputa)

print('Item marcado como Desconocido:', n_item_unknown)
print('Payment Method marcado como Desconocido:', n_payment_unknown)
print('Location marcado como Desconocido:', n_location_unknown)
print('Transaction Date que quedan como NaT:', n_date_missing)


Item marcado como Desconocido: 495
Payment Method marcado como Desconocido: 3168
Location marcado como Desconocido: 3952
Transaction Date que quedan como NaT: 460


## 4. Validación de la base limpia

In [16]:
print('Nulos por columna:')
print(df_clean.isna().sum())
print()
print('Tipos de datos:')
print(df_clean.dtypes)


Nulos por columna:
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

Tipos de datos:
Transaction ID                 str
Item                           str
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object


In [17]:
# Verifica que la relación Total Spent = Quantity * Price Per Unit se cumpla
# en el 100% de las filas, incluidas las reconstruidas.
check = (df_clean['Quantity'] * df_clean['Price Per Unit']).round(2) == df_clean['Total Spent'].round(2)
print('Filas consistentes:', check.sum(), '/', len(df_clean))
assert check.all(), 'Hay filas con Total Spent inconsistente'


Filas consistentes: 9974 / 9974


In [18]:
df_clean.describe(include='all')


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,9974,9974,9974.000000,9974.000000,9974.000000,9974,9974,9514
unique,9974,9,NaN,NaN,NaN,4,3,NaN
top,TXN_1961373,Coffee,NaN,NaN,NaN,Desconocido,Desconocido,NaN
freq,1,1279,NaN,NaN,NaN,3168,3952,NaN
mean,NaN,NaN,3.024865,2.946962,8.927411,NaN,NaN,2023-07-01 23:14:35.593861
min,NaN,NaN,1.000000,1.000000,1.000000,NaN,NaN,2023-01-01 00:00:00
25%,NaN,NaN,2.000000,2.000000,4.000000,NaN,NaN,2023-04-01 00:00:00
50%,NaN,NaN,3.000000,3.000000,8.000000,NaN,NaN,2023-07-02 00:00:00
75%,NaN,NaN,4.000000,4.000000,12.000000,NaN,NaN,2023-10-02 00:00:00
max,NaN,NaN,5.000000,5.000000,25.000000,NaN,NaN,2023-12-31 00:00:00


## 5. Exportar la base limpia

In [19]:
df_clean.to_csv('cafe_sales_clean.csv', index=False)
print('Archivo exportado: cafe_sales_clean.csv')
df_clean.head(10)


Archivo exportado: cafe_sales_clean.csv


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Desconocido,Desconocido,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,Desconocido,2023-03-31
6,TXN_4433211,Desconocido,3,3.0,9.0,Desconocido,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,Desconocido,2023-10-28
8,TXN_4717867,Desconocido,5,3.0,15.0,Desconocido,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,Desconocido,In-store,2023-12-31


## 6. Tabla resumen de problemas y decisiones

Ver también `tabla_resumen.csv` y el `README.md` del repositorio.


In [20]:
resumen = pd.DataFrame([
    {
        'Problema encontrado': 'Valores faltantes (NaN, "ERROR", "UNKNOWN") en Item, Quantity, '
                                'Price Per Unit y Total Spent',
        'Registros afectados': 969 + 479 + 533 + 502,
        'Acción realizada': 'Estandarizar marcadores a NaN; reconstruir con el mapa Item→Precio '
                             'y con Total=Quantity×Price cuando había suficiente información; '
                             'el resto se marcó "Desconocido" o se eliminó (26 filas).',
        'Justificación': 'Reconstruir a partir de relaciones reales de los datos preserva '
                          'información verdadera en lugar de inventarla con la media/moda.',
    },
    {
        'Problema encontrado': 'Valores faltantes en Payment Method y Location',
        'Registros afectados': 3178 + 3961,
        'Acción realizada': 'Se mantienen las filas y se marca la categoría como "Desconocido".',
        'Justificación': 'Con ~32%-40% de datos faltantes, imputar con la moda introduciría un '
                          'sesgo fuerte hacia una sola categoría; es más honesto declarar el dato '
                          'como desconocido.',
    },
    {
        'Problema encontrado': 'Valores faltantes en Transaction Date',
        'Registros afectados': 460,
        'Acción realizada': 'Se dejan como NaT, sin imputar.',
        'Justificación': 'No existe otra columna de la que se pueda derivar la fecha; inventar una '
                          'fecha distorsionaría cualquier análisis temporal.',
    },
    {
        'Problema encontrado': 'Duplicados',
        'Registros afectados': 0,
        'Acción realizada': 'Se verificó con duplicated() sobre todas las columnas y sobre '
                             'Transaction ID; no se encontraron duplicados, no se requirió acción.',
        'Justificación': 'No aplicar una técnica de limpieza que no es necesaria evita alterar '
                          'datos válidos sin motivo.',
    },
    {
        'Problema encontrado': 'Tipos de datos incorrectos (Quantity, Price Per Unit, Total Spent '
                                'y Transaction Date cargados como texto)',
        'Registros afectados': 10000,
        'Acción realizada': 'Conversión con pd.to_numeric()/pd.to_datetime() usando errors="coerce" '
                             'después de limpiar los marcadores "ERROR"/"UNKNOWN".',
        'Justificación': 'Convertir antes de limpiar los marcadores de texto habría fallado o '
                          'generado NaN donde en realidad podía recuperarse el dato.',
    },
    {
        'Problema encontrado': 'Valores atípicos',
        'Registros afectados': 0,
        'Acción realizada': 'Se revisaron los rangos de Quantity (1-5) y Price Per Unit (1.0-5.0) '
                             'tras la conversión a numérico; no se encontraron valores fuera de lo '
                             'razonable para una cafetería.',
        'Justificación': 'No se eliminó ningún dato como atípico porque no había evidencia de '
                          'valores anómalos; eliminar sin evidencia habría sido arbitrario.',
    },
])
resumen.to_csv('tabla_resumen.csv', index=False)
resumen


,Problema encontrado,Registros afectados,Acción realizada,Justificación
0,"Valores faltantes (NaN, ""ERROR"", ""UNKNOWN"") en...",2483,Estandarizar marcadores a NaN; reconstruir con...,Reconstruir a partir de relaciones reales de l...
1,Valores faltantes en Payment Method y Location,7139,Se mantienen las filas y se marca la categoría...,"Con ~32%-40% de datos faltantes, imputar con l..."
2,Valores faltantes en Transaction Date,460,"Se dejan como NaT, sin imputar.",No existe otra columna de la que se pueda deri...
3,Duplicados,0,Se verificó con duplicated() sobre todas las c...,No aplicar una técnica de limpieza que no es n...
4,"Tipos de datos incorrectos (Quantity, Price Pe...",10000,Conversión con pd.to_numeric()/pd.to_datetime(...,Convertir antes de limpiar los marcadores de t...
5,Valores atípicos,0,Se revisaron los rangos de Quantity (1-5) y Pr...,No se eliminó ningún dato como atípico porque ...
